# YOLO Model Inspector
This notebook displays detailed information about a YOLO model including:
- Model architecture
- Class labels/names
- Input size
- Number of parameters
- Layer information

In [ ]:
# Cross-Platform Environment Setup
import os
import sys

def setup_environment():
    """Detects platform and sets up the environment."""
    in_colab = 'google.colab' in sys.modules
    in_kaggle = os.environ.get('KAGGLE_URL_BASE') is not None
    
    if in_colab or in_kaggle:
        print(f"Running on {'Google Colab' if in_colab else 'Kaggle'}. Installing dependencies...")
        !pip install -qU ultralytics wandb roboflow python-dotenv supervision easyocr cvzone
        
        if in_colab:
            from google.colab import userdata
            os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY') or ''
            os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY') or ''
        elif in_kaggle:
            try:
                from kaggle_secrets import UserSecretsClient
                user_secrets = UserSecretsClient()
                os.environ['WANDB_API_KEY'] = user_secrets.get_secret('WANDB_API_KEY')
                os.environ['ROBOFLOW_API_KEY'] = user_secrets.get_secret('ROBOFLOW_API_KEY')
            except Exception:
                print("Kaggle secrets not found. Please set them in the Add-ons menu.")
    else:
        print("Running locally. Loading environment...")
        try:
            from dotenv import load_dotenv
            load_dotenv('../.env')
        except ImportError:
            print("python-dotenv not found. Install it with: pip install python-dotenv")

    # Verify keys
    if not os.environ.get('WANDB_API_KEY'):
        print("Warning: WANDB_API_KEY not set.")
    if not os.environ.get('ROBOFLOW_API_KEY'):
        print("Warning: ROBOFLOW_API_KEY not set.")

setup_environment()

In [ ]:
from ultralytics import YOLO
import torch

# Change this to your model path
MODEL_PATH = "../models/accident_classifier_and_detection.pt"  # Update this path to your model

In [ ]:
# Load the model
model = YOLO(MODEL_PATH)
print(f"✅ Model loaded successfully from: {MODEL_PATH}")

## Class Labels / Names
The classes this model can detect:

In [ ]:
# Get class names/labels
class_names = model.names
print(f"📋 Number of classes: {len(class_names)}\n")
print("Class ID -> Class Name:")
print("-" * 30)
for class_id, class_name in class_names.items():
    print(f"  {class_id:3d} -> {class_name}")

## Model Information
General model details and architecture:

In [ ]:
# Model info
print("🔧 MODEL INFORMATION")
print("=" * 50)
print(f"Task:          {model.task}")
print(f"Model Type:    {type(model.model).__name__}")
print(f"Device:        {'CUDA' if torch.cuda.is_available() else 'CPU'}")

# Get model parameters count
total_params = sum(p.numel() for p in model.model.parameters())
trainable_params = sum(p.numel() for p in model.model.parameters() if p.requires_grad)

print(f"\n📊 PARAMETERS")
print("-" * 50)
print(f"Total Parameters:     {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")
print(f"Model Size (MB):      {total_params * 4 / (1024**2):.2f}")

## Input / Output Details

In [ ]:
# Input/Output details from model args
print("📥 INPUT/OUTPUT DETAILS")
print("=" * 50)

# Try to get model arguments
if hasattr(model.model, 'args'):
    args = model.model.args
    print(f"Input Size (imgsz): {args.get('imgsz', 'N/A')}")
    print(f"Batch Size:         {args.get('batch', 'N/A')}")
    
# Get stride info
if hasattr(model.model, 'stride'):
    print(f"Stride:             {model.model.stride}")

# Number of classes from model head
if hasattr(model.model, 'nc'):
    print(f"Number of Classes:  {model.model.nc}")

## Model Architecture (Layers)

In [ ]:
# Print model architecture summary
print("🏗️ MODEL ARCHITECTURE")
print("=" * 50)
model.info(detailed=False)

## Complete Model Summary (All Layers)

In [ ]:
# Detailed layer-by-layer breakdown
print("📜 DETAILED LAYER INFO")
print("=" * 70)
model.info(detailed=True, verbose=True)

## Export Class Names to JSON
Save the class labels for use in other parts of the project:

In [ ]:
import json
from pathlib import Path

# Export class names to JSON
output_path = Path(MODEL_PATH).stem + "_classes.json"
with open(output_path, 'w') as f:
    json.dump(model.names, f, indent=2)
    
print(f"✅ Class names exported to: {output_path}")
print(f"\nContent:")
print(json.dumps(model.names, indent=2))